# 問題: 飛行機の遅延を予測する

このノートブックの目標は、以下のとおりです:
- ダウンロードした .zip ファイルを処理してデータセットを作成する
- 探索的データ分析 (EDA) を行う
- ベースラインモデルを確立する
- シンプルなモデルからアンサンブルモデルに移行する
- ハイパーパラメータの最適化を実施する
- 特徴量の重要度を確認する


## ビジネスシナリオの概要

あなたは勤務先の会社で、旅行予約ウェブサイトの運営を担当しています。このウェブサイトでは、フライトが遅延した場合のカスタマーエクスペリエンスを向上させたいと考えています。同社では、新しく機能を作成して、お客様に、米国国内便を利用する際、発着便数が多く、非常に混雑している空港を発着するフライトの予約時に、天候によるフライト遅延の有無を知らせたいと考えています。

今回のタスクは、天候によりフライトが遅延するかどうかを機械学習 (ML) で特定し、この問題の一端を解決することです。データセットとして、大手航空会社が運航した国内便の定刻パフォーマンスに関するものを利用できます。このデータを使用して ML モデルをトレーニングし、発着便数の非常に多い空港でフライトが遅延するかどうかを予測します。


## このデータセットについて

データセットには、予定離着陸時刻と実際の離着陸時刻が含まれます。これは、米国内の予定旅客輸送収益の 1% 以上を占める、米国の認可航空会社から報告されたものです。データは、米国運輸統計局 (BTS) の航空情報庁が収集しました。データセットは、2013 年から 2018 年のフライトの日付、飛行時間、出発地、目的地、航空会社、飛行距離、遅延ステータスで構成されます。


### 特徴量

データセットの特徴量の詳細については、[On-time delay dataset features](https://www.transtats.bts.gov/Fields.asp) を参照してください。

### データセットの属性  
ウェブサイト: https://www.transtats.bts.gov/

このラボで使用するデータセットは、米国運輸統計局の航空情報庁でまとめられたものであり、航空機の定刻パフォーマンスデータは https://www.transtats.bts.gov/DatabaseInfo.asp?DB_ID=120&amp;DB_URL=Mode_ID=1&amp;Mode_Desc=Aviation&amp;Subject_ID2=0 で確認できます。

# ステップ 1: 問題の定式化とデータ収集

このプロジェクトを始めるにあたり、このシナリオにおけるビジネス上の問題と達成するビジネス目標を 2、3 文にまとめます。次のセクションでアイデアを書き留めることができます。これには、チームが目指す必要のあるビジネスメトリクスを含めます。その情報を定義したら、ML 問題文を記述します。最後に、このアクティビティが表す ML のタイプに関するコメントを少し追加します。

#### <span style="color: blue;">プロジェクトプレゼンテーション: これらの詳細の要約をプロジェクトプレゼンテーションに含めます。</span>

### 1.ML がこのシナリオのためにデプロイすべきソリューションか、その是非と理由をはっきりさせます。

In [ ]:
# Write your answer here

### 2.ビジネス上の問題、成功のメトリクス、求める ML 出力を定式化します。

In [ ]:
# Write your answer here

### 3.取り扱う ML 問題のタイプを特定します。

In [ ]:
# Write your answer here

### 4.取り扱うデータの適切性を分析します。

In [ ]:
# Write your answer here

### セットアップ

着目点を定めたら、問題の解決を開始できるよう、このラボをセットアップします。

**注:** このノートブックは 25 GB のストレージの `ml.m4.xlarge` ノートブックインスタンスで、作成およびテストされました。

In [ ]:
import os
from pathlib2 import Path
from zipfile import ZipFile
import time

import pandas as pd
import numpy as np
import subprocess

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
instance_type='ml.m4.xlarge'

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# ステップ 2: データの前処理と可視化  
このデータの前処理フェーズでは、データを探索して可視化し、データに対する理解を深めます。まず、必要なライブラリをインポートし、データを pandas の DataFrame に読み込みます。データをインポートした後、データセットを探索します。データセットのシェイプを探し、取り扱う列と列のタイプ (数値、カテゴリー) を調査します。特徴量の平均と範囲をつかむために、特徴量に対する基本的な統計処理を検討します。ターゲット列を精査して、その分布を判断します。


### 考慮すべき具体的な質問

ラボのこのセクションでは、次の質問について検討します。

1. 特徴量に対して実行した基本的な統計から、どのようなことを推測できますか? 
2. ターゲットクラスの分布から、どのようなことを推測できますか?
3. データを探索することで推測できることは他にありますか?

#### <span style="color: blue;">プロジェクトプレゼンテーション: これらの質問や他の同様の質問に対する自分の回答の要約をプロジェクトプレゼンテーションに含めます。</span>

まず、Amazon Simple Storage Service (Amazon S3) のパブリックバケットから、このノートブック環境にデータセットを取り込みます。

In [ ]:
# download the files

zip_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
base_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
csv_base_path = '/home/ec2-user/SageMaker/project/data/csvFlightDelays/'

!mkdir -p {zip_path}
!mkdir -p {csv_base_path}
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data/ {zip_path} --recursive


In [ ]:
zip_files = [str(file) for file in list(Path(base_path).iterdir()) if '.zip' in str(file)]
len(zip_files)

.zip ファイルからカンマ区切り値 (CSV) ファイルを抽出します。

In [ ]:
def zip2csv(zipFile_name , file_path):
    """
    Extract csv from zip files
    zipFile_name: name of the zip file
    file_path : name of the folder to store csv
    """

    try:
        with ZipFile(zipFile_name, 'r') as z: 
            print(f'Extracting {zipFile_name} ') 
            z.extractall(path=file_path) 
    except:
        print(f'zip2csv failed for {zipFile_name}')

for file in zip_files:
    zip2csv(file, csv_base_path)

print("Files Extracted")

In [ ]:
csv_files = [str(file) for file in list(Path(csv_base_path).iterdir()) if '.csv' in str(file)]
len(csv_files)

CSV ファイルをロードする前に、展開したフォルダから HTML ファイルを読み取ります。この HTML ファイルには、データセットに含まれている特徴量の背景を始めとする情報が含まれています。

In [ ]:
from IPython.display import IFrame

IFrame(src=os.path.relpath(f"{csv_base_path}readme.html"), width=1000, height=600)

#### サンプル CSV をロードする

すべての CSV ファイルを結合する前に、単一の CSV ファイルに含まれるデータを調べます。pandas を使用して、最初に `On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv` ファイルを読み取ります。Python の組み込み関数 `read_csv` ([pandas.read_csv ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)) を使用できます。

In [ ]:
df_temp = pd.read_csv(f"{csv_base_path}On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv")

**問題**: データセット内の行と列の長さ、および列名を出力してください。

**ヒント**: DataFrame の行と列を表示するには、`<DataFrame>.shape` 関数を使用します。列名を表示するには、`<DataFrame>.columns` 関数を使用します。

In [ ]:
df_shape = # **ENTER YOUR CODE HERE**
print(f'Rows and columns in one CSV file is {df_shape}')

**問題**: データセットの最初の 10 行を出力してください。 

**ヒント**: `x` の行数を出力するには、pandas の組み込み関数 `head(x)` を使用します。

In [ ]:
# Enter your code here

**問題**: データセット内の列をすべて出力してください。列名を表示するには、`<DataFrame>.columns` を使用します。

In [ ]:
print(f'The column names are :')
print('#########')
for col in <CODE>:# **ENTER YOUR CODE HERE**
    print(col)

**問題**: データセットから、*Del* という単語を含む列をすべて出力してください。この操作によって、*遅延データ* がある列の数を確認できます。

**ヒント**: 特定の `if` ステートメントの条件を満たす値を含めるには、Python のリスト内包表記を使用できます。

例: `[x for x in [1,2,3,4,5] if x > 2]`  

**ヒント**: `in` キーワード ([Python in Keyword documentation](https://www.w3schools.com/python/ref_keyword_in.asp)) を使用して、値がリストにあるかどうかを確認できます。

例: `5 in [1,2,3,4,5]`

In [ ]:
# Enter your code here

データセットの中身を知るのに役立つ質問を以下にいくつか紹介します。

**質問**   

1. データセットに行と列はそれぞれいくつありますか?   
2. データセットに何年分のデータが含まれていますか?   
3. データセットが対象とする期間はどのくらいですか?   
4. データセットにどの航空会社が含まれていますか?   
5. どの発着空港が対象となっていますか?

**ヒント**
- DataFrame の次元を表示するには、`df_temp.shape` を使用します。
- 特定の列を参照するには、`df_temp.columnName` (`df_temp.CarrierDelay` など) を使用します。
- 列に含まれる固有値を取得するには、`df_temp.column.unique()` (`df_temp.Year.unique()` など) を使用します。

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", <CODE>)
print("The months covered in this dataset are: ", <CODE>)
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

**問題**: 発着空港の総数はいくつですか?

**ヒント**: **Origin** と **Dest** の列を使って各空港の値を割り出すには、pandas の `values_count` 関数 ([pandas.Series.value_counts ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)) を使えます。

In [ ]:
counts = pd.DataFrame({'Origin':<CODE>, 'Destination':<CODE>})
counts

**問題**: データセット内のフライト数に基づいて、上位 15 件の発着空港を出力してください。

**ヒント**: pandas の `sort_values` 関数 ([pandas.DataFrame.sort_values ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html)) を使用できます。

In [ ]:
counts.sort_values(by=<CODE>,ascending=False).head(15) # Enter your code here

**フライトに関するすべての情報があれば、そのフライトが遅延するかどうかを予測できますか?**

**ArrDel15** 列は、遅延が 15 分を超える場合に値 *1* を取る指標変数です。それ以外の場合は、*0* の値を取ります。

この変数を分類問題のターゲット列として使用できます。

たとえば、出張でサンフランシスコからロサンゼルスへ向かうとします。ロサンゼルスでの予約をうまく手配したいと考えています。したがって、一連の特徴量に基づいて、フライトが遅延するかどうか知りたいと考えています。このデータセットには、フライトの前に確認しておきたい特徴量はいくつありますか?

`DepDelay`、`ArrDelay`、`CarrierDelay`、`WeatherDelay`、`NASDelay`、`SecurityDelay`、`LateAircraftDelay`、`DivArrDelay` などの列には、遅延に関する情報が含まれています。ただし、この遅延が出発地、目的地のどちらで発生したかは不明です。着陸 10 分前に天候の急変で遅延が発生していた場合、このデータはロサンゼルスでの予約をうまく扱うのには役立ちません。

そこで、問題文をシンプルにするため、以下の列を考慮に入れて到着遅延を予測します:<br>

`Year`、`Quarter`、`Month`、`DayofMonth`、`DayOfWeek`、`FlightDate`、`Reporting_Airline`、`Origin`、`OriginState`、`Dest`、`DestState`、`CRSDepTime`、`DepDelayMinutes`、`DepartureDelayGroups`、`Cancelled`、`Diverted`、`Distance`、`DistanceGroup`、`ArrDelay`、`ArrDelayMinutes`、`ArrDel15`、`AirTime`

また、以下のように発着空港をフィルタリングします。
- 主要空港: ATL、ORD、DFW、DEN、CLT、LAX、IAH、PHX、SFO
-トップ 5 の航空会社：UA、OO、WN、AA、DL

この情報は、結合する CSV ファイル全体のデータサイズを縮小するのに役立ちます。

#### すべての CSV ファイルを結合する
 
まず、各ファイルからの個々の DataFrame をコピーするために使用する空の DataFrame を作成します。次に、`csv_files` リスト内の各ファイルに対して以下を実行します。

1. DataFrame に CSV ファイルを読み込みます。 
2. `filter_cols` 変数に基づいて列をフィルタリングします。

```
        columns = ['col1', 'col2']
        df_filter = df[columns]
```

3. 各 `subset_vals` には `subset_cols` だけを保持します。pandas の `isin` 関数 ([pandas.DataFram.isin ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isin.html)) を使用して、`val` が DataFrame の列にあるかどうかを確認します。次に、それを含む行を選択します。

```
        df_eg[df_eg['col1'].isin('5')]
```

4. DataFrame を空の DataFrame と連結します。 

In [ ]:
def combine_csv(csv_files, filter_cols, subset_cols, subset_vals, file_name):

    """
    Combine csv files into one Data Frame
    csv_files: list of csv file paths
    filter_cols: list of columns to filter
    subset_cols: list of columns to subset rows
    subset_vals: list of list of values to subset rows
    """

    df = pd.DataFrame()
    
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_temp = df_temp[filter_cols]
        for col, val in zip(subset_cols,subset_vals):
            df_temp = df_temp[df_temp[col].isin(val)]      
        
        df = pd.concat([df, df_temp], axis=0)
      
    df.to_csv(file_name, index=False)
    print(f'Combined csv stored at {file_name}')

In [ ]:
#cols is the list of columns to predict Arrival Delay 
cols = ['Year','Quarter','Month','DayofMonth','DayOfWeek','FlightDate',
        'Reporting_Airline','Origin','OriginState','Dest','DestState',
        'CRSDepTime','Cancelled','Diverted','Distance','DistanceGroup',
        'ArrDelay','ArrDelayMinutes','ArrDel15','AirTime']

subset_cols = ['Origin', 'Dest', 'Reporting_Airline']

# subset_vals is a list collection of the top origin and destination airports and top 5 airlines
subset_vals = [['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['UA', 'OO', 'WN', 'AA', 'DL']]

前述の関数を使用して異なるファイルをすべて単一のファイルにマージして、簡単に読み取れるようにします。

**注**: この処理が完了するまで 5～7 分かかります。

In [ ]:
start = time.time()
combined_csv_filename = f"{base_path}combined_files.csv"
combine_csv(csv_files, cols, subset_cols, subset_vals, combined_csv_filename)
print(f'CSVs merged in {round((time.time() - start)/60,2)} minutes')

# データセットをロードする

結合されたデータセットをロードします。

In [ ]:
data = pd.read_csv(combined_csv_filename)

最初の 5 つのレコードを出力します。

In [ ]:
# Enter your code here 

データセットの中身を知るのに役立つ質問を以下にいくつか紹介します。

**質問**   

1. データセットに行と列はそれぞれいくつありますか?   
2. データセットに何年分のデータが含まれていますか?   
3. データセットが対象とする期間はどのくらいですか?   
4. データセットにどの航空会社が含まれていますか?   
5. どの発着空港が対象となっていますか?

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", list(<CODE>))
print("The months covered in this dataset are: ", sorted(list(<CODE>)))
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

ターゲット列を **is_delay** (*1* は到着時間の遅れが 15 分を超えたことを意味し、*0* は残りすべてのケースを意味します) と定義します。`rename` メソッドを使用して、列名を **ArrDel15** から **is_delay** に変更します。

**ヒント**: pandas の `rename` 関数 ([pandas.DataFrame.rename ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)) を使用できます。

例:
```
data.rename(columns={'col1':'column1'}, inplace=True)
```

In [ ]:
data.rename(columns=<CODE>, inplace=True) # Enter your code here

すべての列にわたって null を探します。`isnull()` 関数 ([pandas.isnull ドキュメント](https://pandas.pydata.org/pandas-docs/version/0.17.0/generated/pandas.isnull.html)) を使用できます。

**ヒント**: `isnull()` を使用すると、特定の値が null かどうかを検出できます。これは、その場所にブール値 (*True* または *False*) を返します。列数を集計するには、`sum(axis=0)` 関数 (`df.isnull().sum(axis=0)` など) を使用します。

In [ ]:
# Enter your code here

1,658,130 行のうち、1.3 パーセントに当たる 22,540 行に、到着遅延の詳細と飛行時間が含まれていません。これらの行を削除または補完できます。ドキュメントには、欠落行については一切記載されていません。


In [ ]:
### Remove null columns
data = data[~data.is_delay.isnull()]
data.isnull().sum(axis = 0)

CRSDepTime から 24 時間形式の時刻を取得します。

In [ ]:
data['DepHourofDay'] = (data['CRSDepTime']//100)

## **ML の問題文**
- 一連の特徴量に基づいて、フライトが 15 分を超えて遅延するかどうかを予測できますか?
- ターゲット変数に入る値は *0* または *1* に限られるため、分類アルゴリズムを使用できます。

モデリングに進む前に、特徴量の分布、相関などを確認することをお勧めします。
- これで、データ中の非線形性またはパターンがわかる
    - 線形モデル: 検出力のある、指数関数的な、あるいは相互作用的な特徴量を追加する
    - 非線形モデルを試す
- データの不均衡 
    - モデルのパフォーマンス (曲線下面積/AUC に対する正解率) に偏りが生じないメトリクスを選択する
    - 加重/カスタム損失関数を使用する
- 欠損データ
    - 単純な統計に基づいて補完を行う - 平均値、中央値、最頻値 (数値変数)、頻度クラス (カテゴリー変数)
    - クラスターベースの補完 (k 近傍法/KNN で列の値を予測)
    - 列を削除する

### データ探索

*delay* と *no delay* のクラスを見比べます。


In [ ]:
(data.groupby('is_delay').size()/len(data) ).plot(kind='bar')# Enter your code here
plt.ylabel('Frequency')
plt.title('Distribution of classes')
plt.show()

**質問**: *delay* と *no delay* の比率に関する棒グラフからどのようなことを推測できますか?

In [ ]:
# Enter your answer here

次のセルを実行して、質問に答えてみましょう。

In [ ]:
viz_columns = ['Month', 'DepHourofDay', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest']
fig, axes = plt.subplots(3, 2, figsize=(20,20), squeeze=False)
# fig.autofmt_xdate(rotation=90)

for idx, column in enumerate(viz_columns):
    ax = axes[idx//2, idx%2]
    temp = data.groupby(column)['is_delay'].value_counts(normalize=True).rename('percentage').\
    mul(100).reset_index().sort_values(column)
    sns.barplot(x=column, y="percentage", hue="is_delay", data=temp, ax=ax)
    plt.ylabel('% delay/no-delay')
    

plt.show()

In [ ]:
sns.lmplot( x="is_delay", y="Distance", data=data, fit_reg=False, hue='is_delay', legend=False)
plt.legend(loc='center')
plt.xlabel('is_delay')
plt.ylabel('Distance')
plt.show()

**質問**

先ほど見たグラフのデータを使用して、次の質問に答えます。

- 遅延が最も多いのは何月ですか?
- 遅延が最も多いのは何時ですか?
- 遅延が最も多いのは何曜日ですか?
- 遅延が最も多いのはどの航空会社ですか?
- 遅延が最も多い発着空港はどこですか?
- 飛行距離は遅延の要因ですか?

In [ ]:
# Enter your answers here

### 特徴量

列と、列固有のタイプをすべて確認します。

In [ ]:
data.columns

In [ ]:
data.dtypes

必須の列を以下のようにフィルタリングします。
- 日付を表す列には *Year*、*Quarter*、*Month*、*DayofMonth*、*DayOfWeek* があるため、*Date* は冗長です。
- *OriginState* と *DestState* の代わりに、*Origin* コードと *Dest* コードを使用します。
- フライトが遅延するかどうかを分類するだけであるため、*TotalDelayMinutes*、*DepDelayMinutes*、*ArrDelayMinutes* は必要ありません。

*DepHourofDay* はターゲットとの定量的関係がないため、カテゴリー変数として扱います。
- この変数のワンホットエンコーディング (ダミー変数) を必要とする場合は、23 列増えることになります。
- カテゴリー変数を扱うその他の方法として、ハッシュエンコーディング、平均値正規化エンコーディング、値のバケット化などがあります。
- この場合、バケットに分割するだけで済みます。

列タイプをカテゴリーに変更するには、`astype` 関数 ([pandas.DataFrame.astype ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.astype.html)) を使用します。

In [ ]:
data_orig = data.copy()
data = data[[ 'is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay']]
categorical_columns  = ['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'DepHourofDay']
for c in categorical_columns:
    data[c] = data[c].astype('category')

ワンホットエンコーディングを使用するには、選択したカテゴリー列に `get_dummies` 関数を使用します。次に、pandas の `concat` 関数を使用して、これらの生成された特徴量を元のデータセットに連結できます。カテゴリー変数のエンコーディングには、キーワード `drop_first=True` を使用して*ダミーエンコーディング*を使うこともできます。ダミーエンコーディングの詳細については、[Dummy variable (statistics)](https://en.wikiversity.org/wiki/Dummy_variable_(statistics)) を参照してください。

例:
```
pd.get_dummies(df[['column1','columns2']], drop_first=True)
```

In [ ]:
data_dummies = pd.get_dummies(<CODE>, drop_first=True) # Enter your code here
data = pd.concat([<CODE>, <CODE>], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

データセットの長さと新しい列を確認します。

**ヒント**: `shape` と `columns` のプロパティを使用します。

In [ ]:
# Enter your code here

In [ ]:
# Enter your code here

これで、モデルをトレーニングする準備が整いました。データを分割する前に、**is_delay** 列の名前を *target* に変更します。

**ヒント**: pandas の `rename` 関数 ([pandas.DataFrame.rename ドキュメント](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)) を使用できます。

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

## <span style="color:red"> ステップ 2 の終了 </span>

ローカルコンピュータにプロジェクトファイルを保存します。次の一連のステップを実行します。

1. 左側のファイルエクスプローラーで、作業中のノートブックを右クリックします。

2. **ダウンロード** を選択し、ファイルをローカルに保存します。 

この操作により、現在のノートブックがコンピュータのデフォルトのダウンロードフォルダにダウンロードされます。

# ステップ 3: モデルのトレーニングと評価

データセットを DataFrame から機械学習アルゴリズムで使用できる形式に変換するときに、いくつかの準備ステップが必要になります。Amazon SageMaker では、以下のステップを実行する必要があります。

1. `sklearn.model_selection.train_test_split` を使用して、データを `train_data`、`validation_data`、`test_data` に分割します。 

2. Amazon SageMaker トレーニングジョブで使用できる適切なファイル形式にデータセットを変換します。これは CSV ファイルまたは record protobuf のいずれかです。詳細については、[トレーニングの共通データ形式](https://docs.aws.amazon.com/sagemaker/latest/dg/cdf-training.html) を参照してください。 

3. S3 バケットにデータをアップロードします。バケットを作成したことがない場合は、[バケットの作成](https://docs.aws.amazon.com/AmazonS3/latest/gsg/CreatingABucket.html) を参照してください。 

以下のセルを使用して、これらのステップを遂行します。必要に応じてセルを挿入、削除します。

#### <span style="color: blue;">プロジェクトプレゼンテーション: このフェーズでの主な決定事項をプロジェクトプレゼンテーションに記録します。</span>

### トレーニングとテストの分割

In [ ]:
from sklearn.model_selection import train_test_split
def split_data(data):
    train, test_and_validate = train_test_split(data, test_size=0.2, random_state=42, stratify=data['target'])
    test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])
    return train, validate, test

In [ ]:
train, validate, test = split_data(data)
print(train['target'].value_counts())
print(test['target'].value_counts())
print(validate['target'].value_counts())

**解答例**
```
0.0    1033570
1.0     274902
Name: target, dtype: int64
0.0    129076
1.0     34483
Name: target, dtype: int64
0.0    129612
1.0     33947
Name: target, dtype: int64
```

### ベースライン分類モデル

In [ ]:
import sagemaker
from sagemaker.serializers import CSVSerializer
from sagemaker.amazon.amazon_estimator import RecordSet
import boto3

# Instantiate the LinearLearner estimator object with 1 ml.m4.xlarge
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=<CODE>,
                                               instance_type=<CODE>,
                                               predictor_type=<CODE>,
                                               binary_classifier_model_selection_criteria=<CODE>)

### サンプルコード
```
num_classes = len(pd.unique(train_labels))
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                              instance_count=1,
                                              instance_type='ml.m4.xlarge',
                                              predictor_type='binary_classifier',
                                              binary_classifier_model_selection_criteria = 'cross_entropy_loss')
                                              
```

Linear learner (線形学習者) は、protobuf または CSV コンテンツタイプのトレーニングデータに対応します。また、protobuf、CSV、JavaScript Object Notation (JSON) コンテンツタイプでの推論リクエストにも対応します。トレーニングデータが特徴量と正解ラベルを持つのに対し、推論リクエストのデータは特徴量のみを持ちます。

AWS では、本番パイプラインでデータを Amazon SageMaker の protobuf 形式に変換し、Amazon S3 に保存することを推奨します。すばやく開始できるように、AWS では、データセットがローカルメモリに収まるほど小さい場合に `record_set` オペレーションを利用して、変換とアップロードを実行できます。このメソッドは、すでにお持ちのような NumPy 配列に対応しているため、このステップではそれを使ってみましょう。`RecordSet` オブジェクトは、データの一時的な Amazon S3 の場所を追跡します。`estimator.record_set` 関数を使用して、トレーニング、検証、テストのレコードを作成します。次に、`estimator.fit` 関数を使用してトレーニングジョブを開始します。

In [ ]:
### Create train, validate, and test records
train_records = classifier_estimator.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

では、アップロードしたデータセットでモデルをトレーニングします。

### サンプルコード
```
linear.fit([train_records,val_records,test_records])
```

In [ ]:
### Fit the classifier
# Enter your code here

## モデルの評価
このセクションでは、トレーニング済みモデルを評価します。

まず、トレーニングジョブのメトリクスを調べます。

In [ ]:
sagemaker.analytics.TrainingJobAnalytics(classifier_estimator._current_job_name, 
                                         metric_names = ['test:objective_loss', 
                                                         'test:binary_f_beta',
                                                         'test:precision',
                                                         'test:recall']
                                        ).dataframe()

次に、テストデータを Amazon S3 にロードし、バッチ予測関数を使用して、予測の実行に役立つ関数をいくつかセットアップします。バッチ予測を使用すると、提供されたテストデータに対して予測を実行する場合にのみインスタンスが実行されるため、コストの削減に役立ちます。

**注:** `<LabBucketName>`は、ラボのセットアップ中に作成されたラボバケットの名前に置き換えます。

In [ ]:
import io
#bucket='<LabBucketName>'
prefix='flight-linear'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
def batch_linear_predict(test_data, estimator):
    batch_X = test_data.iloc[:,1:];
    batch_X_file='batch-in.csv'
    upload_s3_csv(batch_X_file, 'batch-in', batch_X)

    batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
    batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

    classifier_transformer = estimator.transformer(instance_count=1,
                                           instance_type='ml.m4.xlarge',
                                           strategy='MultiRecord',
                                           assemble_with='Line',
                                           output_path=batch_output)

    classifier_transformer.transform(data=batch_input,
                             data_type='S3Prefix',
                             content_type='text/csv',
                             split_type='Line')
    
    classifier_transformer.wait()

    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
    target_predicted_df = pd.read_json(io.BytesIO(obj['Body'].read()),orient="records",lines=True)
    return test_data.iloc[:,0], target_predicted_df.iloc[:,0]


テストデータセットで予測を実行するには、テストデータセットで (前に定義された) `batch_linear_predict` 関数を実行します。


In [ ]:
test_labels, target_predicted = batch_linear_predict(test, classifier_estimator)

いくつかの関数を作成して、混同行列のプロットや、さまざまなスコアリングメトリクスを表示します。

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(test_labels, target_predicted):
    matrix = confusion_matrix(test_labels, target_predicted)
    df_confusion = pd.DataFrame(matrix)
    colormap = sns.color_palette("BrBG", 10)
    sns.heatmap(df_confusion, annot=True, fmt='.2f', cbar=None, cmap=colormap)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.ylabel("True Class")
    plt.xlabel("Predicted Class")
    plt.show()
    

In [ ]:
from sklearn import metrics

def plot_roc(test_labels, target_predicted):
    TN, FP, FN, TP = confusion_matrix(test_labels, target_predicted).ravel()
    # Sensitivity, hit rate, recall, or true positive rate
    Sensitivity  = float(TP)/(TP+FN)*100
    # Specificity or true negative rate
    Specificity  = float(TN)/(TN+FP)*100
    # Precision or positive predictive value
    Precision = float(TP)/(TP+FP)*100
    # Negative predictive value
    NPV = float(TN)/(TN+FN)*100
    # Fall out or false positive rate
    FPR = float(FP)/(FP+TN)*100
    # False negative rate
    FNR = float(FN)/(TP+FN)*100
    # False discovery rate
    FDR = float(FP)/(TP+FP)*100
    # Overall accuracy
    ACC = float(TP+TN)/(TP+FP+FN+TN)*100

    print("Sensitivity or TPR: ", Sensitivity, "%") 
    print( "Specificity or TNR: ",Specificity, "%") 
    print("Precision: ",Precision, "%") 
    print("Negative Predictive Value: ",NPV, "%") 
    print( "False Positive Rate: ",FPR,"%")
    print("False Negative Rate: ",FNR, "%") 
    print("False Discovery Rate: ",FDR, "%" )
    print("Accuracy: ",ACC, "%") 

    test_labels = test.iloc[:,0];
    print("Validation AUC", metrics.roc_auc_score(test_labels, target_predicted) )

    fpr, tpr, thresholds = metrics.roc_curve(test_labels, target_predicted)
    roc_auc = metrics.auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic')
    plt.legend(loc="lower right")

    # create the axis of thresholds (scores)
    ax2 = plt.gca().twinx()
    ax2.plot(fpr, thresholds, markeredgecolor='r',linestyle='dashed', color='r')
    ax2.set_ylabel('Threshold',color='r')
    ax2.set_ylim([thresholds[-1],thresholds[0]])
    ax2.set_xlim([fpr[0],fpr[-1]])

    print(plt.figure())

混同行列をプロットするには、バッチジョブから `test_labels` および `target_predicted` のデータに対して `plot_confusion_matrix` 関数を呼び出します。

In [ ]:
# Enter your code here

統計情報を出力し、受信者動作特性 (ROC) 曲線をプロットするには、バッチジョブから `test_labels` および `target_predicted` のデータに対して `plot_roc` 関数を呼び出します。

In [ ]:
# Enter your code here

### 考慮すべき重要な質問:

1. テストセットでのモデルのパフォーマンスは、トレーニングセットでのパフォーマンスと比べて、どう違いますか? この比較からどのようなことを推測できますか? 
2. 正解率、適合率、再現率などのメトリクスの結果に明らかな違いはありますか? ある場合、そのような違いが見られる理由は何だと思われますか? 
3. ビジネスの状況と目標を考えると、ここで考慮すべき最も重要なメトリクスはどれですか? それはなぜですか?
4. 最も重要だと考えるメトリクスの結果は、ビジネス観点でのニーズを十分に満たすものですか? そうでない場合、次のイテレーションで変更できるのはどんなことですか? (これは、次の特徴量エンジニアリングのセクションで発生します)。

以下のセルを使用して、これらの質問や他の質問に答えてください。必要に応じてセルを挿入、削除します。

#### <span style="color: blue;">プロジェクトプレゼンテーション: このセクションで回答するこれらの質問や、他の同様の質問に対する回答をプロジェクトプレゼンテーションに記録します。重要な詳細情報と決定事項を記録します。</span>


**質問**: 混同行列からどのようなことが言えますか?


In [ ]:
# Enter your answer here

## <span style="color:red"> ステップ 3 の終了 </span>

ローカルコンピュータにプロジェクトファイルを保存します。次の一連のステップを実行します。

1. 左側のファイルエクスプローラーで、作業中のノートブックを右クリックします。

2. **ダウンロード** を選択し、ファイルをローカルに保存します。 

この操作により、現在のノートブックがコンピュータのデフォルトのダウンロードフォルダにダウンロードされます。

# イテレーション 2

# ステップ 4: 特徴量エンジニアリング

これで、モデルのトレーニングと評価の 1 回目のイテレーションを完了しました。モデルで最初に得られた結果が、ビジネス上の問題を解決するにはおそらく不十分だったとすると、モデルのパフォーマンスをどうにか改善するには、データに関して何を変更できるでしょうか?

### 考慮すべき重要な質問:

1. 2 つのメインクラス (*delay* と *no delay*) の均衡は、モデルのパフォーマンスにどう影響する可能性がありますか?
2. 相関する特徴量はありますか?
3. この段階で、モデルのパフォーマンスに良い影響を与える可能性があり、実行できる特徴量削減の手法はありますか? 
4. データまたはデータセットの追加を検討できますか?
5. 特徴量エンジニアリングを実行した結果、1 回目のイテレーションと比べて、モデルのパフォーマンスはどうなりますか?

以下のセルを使用し、上記の質問に従って、モデルのパフォーマンスが改善すると考えられる特徴エンジニアリングの手法を実行します (前の質問を参考として使用してください)。必要に応じてセルを挿入、削除します。

#### <span style="color: blue;">プロジェクトプレゼンテーション: このセクションで使用する重要な決定事項と方法をプロジェクトプレゼンテーションに記録します。また、モデルを再評価した後に取得する新しいパフォーマンスメトリクスもすべて含めます。</span>

開始する前に、適合率と再現率が約 80 パーセントであるのに対し、正解率が 99 パーセントである理由を考えてみてください。

特徴量を追加する:

1. 祝日
2. 天候

2014 年から 2018 年の祝日はすべてわかっているため、指標変数 **is_holiday** を作成して祝日をマークできます。

たとえば、祝日期間中は他の日に比べてフライトの遅延率が高い可能性があるとします。2014～2018 年の祝日を含むブール変数 `is_holiday` を追加します。

In [ ]:
# Source: http://www.calendarpedia.com/holidays/federal-holidays-2014.html

holidays_14 = ['2014-01-01',  '2014-01-20', '2014-02-17', '2014-05-26', '2014-07-04', '2014-09-01', '2014-10-13', '2014-11-11', '2014-11-27', '2014-12-25' ] 
holidays_15 = ['2015-01-01',  '2015-01-19', '2015-02-16', '2015-05-25', '2015-06-03', '2015-07-04', '2015-09-07', '2015-10-12', '2015-11-11', '2015-11-26', '2015-12-25'] 
holidays_16 = ['2016-01-01',  '2016-01-18', '2016-02-15', '2016-05-30', '2016-07-04', '2016-09-05', '2016-10-10', '2016-11-11', '2016-11-24', '2016-12-25', '2016-12-26']
holidays_17 = ['2017-01-02', '2017-01-16', '2017-02-20', '2017-05-29' , '2017-07-04', '2017-09-04' ,'2017-10-09', '2017-11-10', '2017-11-23', '2017-12-25']
holidays_18 = ['2018-01-01', '2018-01-15', '2018-02-19', '2018-05-28' , '2018-07-04', '2018-09-03' ,'2018-10-08', '2018-11-12','2018-11-22', '2018-12-25']
holidays = holidays_14+ holidays_15+ holidays_16 + holidays_17+ holidays_18

### Add indicator variable for holidays
data_orig['is_holiday'] = # Enter your code here 

気象データは https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&amp;stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&amp;dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&amp;startDate=2014-01-01&amp;endDate=2018-12-31 から取得しました。
<br>

このデータセットには、都市の風速、降雨量、積雪、気温に関する情報が、空港コード別に含まれています。

**質問**: 降雨、強風、積雪による悪天候がフライトの遅延につながる可能性がありますか? ここで、チェックを行います。

In [ ]:
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data2/daily-summaries.csv /home/ec2-user/SageMaker/project/data/
#!wget 'https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&startDate=2014-01-01&endDate=2018-12-31' -O /home/ec2-user/SageMaker/project/data/daily-summaries.csv

空港コードについて用意されている気象データをデータセットにインポートします。分析には、次のステーションと空港を使用します。気象ステーションを空港名にマップする *airport* という新しい列を作成します。

In [ ]:
weather = pd.read_csv('/home/ec2-user/SageMaker/project/data/daily-summaries.csv')
station = ['USW00023174','USW00012960','USW00003017','USW00094846','USW00013874','USW00023234','USW00003927','USW00023183','USW00013881'] 
airports = ['LAX', 'IAH', 'DEN', 'ORD', 'ATL', 'SFO', 'DFW', 'PHX', 'CLT']

### Map weather stations to airport code
station_map = {s:a for s,a in zip(station, airports)}
weather['airport'] = weather['STATION'].map(station_map)

**DATE** 列から *MONTH* という別の列を作成します。

In [ ]:
weather['MONTH'] = weather['DATE'].apply(lambda x: x.split('-')[1])
weather.head()

### サンプル出力
```
  STATION     DATE      AWND PRCP SNOW SNWD TAVG TMAX  TMIN airport MONTH
0 USW00023174 2014-01-01 16   0   NaN  NaN 131.0 178.0 78.0  LAX    01
1 USW00023174 2014-01-02 22   0   NaN  NaN 159.0 256.0 100.0 LAX    01
2 USW00023174 2014-01-03 17   0   NaN  NaN 140.0 178.0 83.0  LAX    01
3 USW00023174 2014-01-04 18   0   NaN  NaN 136.0 183.0 100.0 LAX    01
4 USW00023174 2014-01-05 18   0   NaN  NaN 151.0 244.0 83.0  LAX    01
```

**SNOW** 列と **SNWD** 列を分析し、`fillna()` を使用して欠損値を処理します。`isna()` 関数を使用して、すべての列について欠損値があるかどうかを確認します。

In [ ]:
weather.SNOW.fillna(0, inplace=True)
weather.SNWD.fillna(0, inplace=True)
weather.isna().sum()

**問題**: *TAVG*、*TMAX*、*TMIN* に欠損値がある行のインデックスを出力してください。

**ヒント**: 欠損値がある行を検索するには、`isna()` 関数を使用します。次に、*idx* 変数のリストを使用して、インデックスを取得します。

In [ ]:
idx = np.array([i for i in range(len(weather))])
TAVG_idx = idx[weather.TAVG.isna()] 
TMAX_idx = # Enter your code here 
TMIN_idx = # Enter your code here 
TAVG_idx

### サンプル出力

```
array([ 3956,  3957,  3958,  3959,  3960,  3961,  3962,  3963,  3964,
        3965,  3966,  3967,  3968,  3969,  3970,  3971,  3972,  3973,
        3974,  3975,  3976,  3977,  3978,  3979,  3980,  3981,  3982,
        3983,  3984,  3985,  4017,  4018,  4019,  4020,  4021,  4022,
        4023,  4024,  4025,  4026,  4027,  4028,  4029,  4030,  4031,
        4032,  4033,  4034,  4035,  4036,  4037,  4038,  4039,  4040,
        4041,  4042,  4043,  4044,  4045,  4046,  4047, 13420])
```

欠落している *TAVG*、*TMAX*、*TMIN* の値に、特定のステーションまたは空港の平均値を代入できます。*TAVG_idx* の連続する行が欠落しているため、直前の値を代入することはできません。代わりに、平均値を代入します。`groupby` 関数を使用して、平均値を含む変数を集計します。

**ヒント:** `MONTH` と `STATION` でグループ化します。

In [ ]:
weather_impute = weather.groupby([<CODE>]).agg({'TAVG':'mean','TMAX':'mean', 'TMIN':'mean' }).reset_index()# Enter your code here
weather_impute.head(2)

平均値データを気象データとマージします。

In [ ]:

weather = pd.merge(weather, weather_impute,  how='left', left_on=['MONTH','STATION'], right_on = ['MONTH','STATION'])\
.rename(columns = {'TAVG_y':'TAVG_AVG',
                   'TMAX_y':'TMAX_AVG', 
                   'TMIN_y':'TMIN_AVG',
                   'TAVG_x':'TAVG',
                   'TMAX_x':'TMAX', 
                   'TMIN_x':'TMIN'})

欠損値があるかどうかをもう一度確認します。

In [ ]:
weather.TAVG[TAVG_idx] = weather.TAVG_AVG[TAVG_idx]
weather.TMAX[TMAX_idx] = weather.TMAX_AVG[TMAX_idx]
weather.TMIN[TMIN_idx] = weather.TMIN_AVG[TMIN_idx]
weather.isna().sum()

データセットから `STATION,MONTH,TAVG_AVG,TMAX_AVG,TMIN_AVG,TMAX,TMIN,SNWD` をドロップします。

In [ ]:
weather.drop(columns=['STATION','MONTH','TAVG_AVG', 'TMAX_AVG', 'TMIN_AVG', 'TMAX' ,'TMIN', 'SNWD'],inplace=True)

データセットに出発地と目的地の気象条件を追加します。

In [ ]:
### Add origin weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Origin'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_O','PRCP':'PRCP_O', 'TAVG':'TAVG_O', 'SNOW': 'SNOW_O'})\
.drop(columns=['DATE','airport'])

### Add destination weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Dest'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_D','PRCP':'PRCP_D', 'TAVG':'TAVG_D', 'SNOW': 'SNOW_D'})\
.drop(columns=['DATE','airport'])

**注意**: 結合後に null/NA を確認することをお勧めします。

In [ ]:
sum(data.isna().any())

In [ ]:
data_orig.columns

ワンホットエンコーディングを使用して、カテゴリーデータを数値データに変換します。

In [ ]:
data = data_orig.copy()
data = data[['is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay','is_holiday', 'AWND_O', 'PRCP_O',
       'TAVG_O', 'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D']]


categorical_columns  = ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']
for c in categorical_columns:
    data[c] = data[c].astype('category')

In [ ]:
data_dummies = pd.get_dummies(data[['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']], drop_first=True)
data = pd.concat([data, data_dummies], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

新しい列を確認します。

In [ ]:
data.shape

In [ ]:
data.columns

### サンプル出力

```
Index(['Distance', 'DepHourofDay', 'is_delay', 'AWND_O', 'PRCP_O', 'TAVG_O',
       'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D', 'Year_2015',
       'Year_2016', 'Year_2017', 'Year_2018', 'Quarter_2', 'Quarter_3',
       'Quarter_4', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12',
       'DayofMonth_2', 'DayofMonth_3', 'DayofMonth_4', 'DayofMonth_5',
       'DayofMonth_6', 'DayofMonth_7', 'DayofMonth_8', 'DayofMonth_9',
       'DayofMonth_10', 'DayofMonth_11', 'DayofMonth_12', 'DayofMonth_13',
       'DayofMonth_14', 'DayofMonth_15', 'DayofMonth_16', 'DayofMonth_17',
       'DayofMonth_18', 'DayofMonth_19', 'DayofMonth_20', 'DayofMonth_21',
       'DayofMonth_22', 'DayofMonth_23', 'DayofMonth_24', 'DayofMonth_25',
       'DayofMonth_26', 'DayofMonth_27', 'DayofMonth_28', 'DayofMonth_29',
       'DayofMonth_30', 'DayofMonth_31', 'DayOfWeek_2', 'DayOfWeek_3',
       'DayOfWeek_4', 'DayOfWeek_5', 'DayOfWeek_6', 'DayOfWeek_7',
       'Reporting_Airline_DL', 'Reporting_Airline_OO', 'Reporting_Airline_UA',
       'Reporting_Airline_WN', 'Origin_CLT', 'Origin_DEN', 'Origin_DFW',
       'Origin_IAH', 'Origin_LAX', 'Origin_ORD', 'Origin_PHX', 'Origin_SFO',
       'Dest_CLT', 'Dest_DEN', 'Dest_DFW', 'Dest_IAH', 'Dest_LAX', 'Dest_ORD',
       'Dest_PHX', 'Dest_SFO', 'is_holiday_1'],
      dtype='object')
```

列名を **is_delay** から *target* に戻します。前に使用したコードと同じコードを使用します。

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

トレーニングセットをもう一度作成します。

**ヒント:** 先ほど定義した (かつ使用した) `split_data` 関数を使用します。

In [ ]:
# Enter your code here

### 新しいベースライン分類器

ここで、上記の新しい特徴量によりモデルの予測検出力が強化されたかどうかを確認します。

In [ ]:
# Instantiate the LinearLearner estimator object
classifier_estimator2 = # Enter your code here

### サンプルコード

```
num_classes = len(pd.unique(train_labels)) 
classifier_estimator2 = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=1,
                                               instance_type='ml.m4.xlarge',
                                               predictor_type='binary_classifier',
                                               binary_classifier_model_selection_criteria = 'cross_entropy_loss')
```

In [ ]:
train_records = classifier_estimator2.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator2.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator2.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

作成したばかりの 3 つのデータセットを使用して、モデルをトレーニングします。

In [ ]:
# Enter your code here

新しくトレーニングされたモデルを使用して、バッチ予測を実行します。

In [ ]:
# Enter your code here

混同行列をプロットします。

In [ ]:
# Enter your code here

ROC 曲線をプロットします。

In [ ]:
# Enter your code here

線形モデルから、パフォーマンスが少ししか改善されなかったことがわかります。Amazon SageMaker で *XGBoost* というツリーベースのアンサンブルモデルを試します。

### XGBoost モデルを試す

以下の一連のステップを実行します。  

1. トレーニングセット変数を使用し、それらを CSV ファイル (train.csv、validation.csv、test.csv) として保存します。
2. 変数にバケット名を格納します。Amazon S3 バケット名は、ラボ手順の左側に表示されます。 
a. `bucket = <LabBucketName>`  
b. `prefix = 'flight-xgb'`  
3. AWS SDK for Python (Boto3) を使用して、モデルをバケットにアップロードします。   

In [ ]:
bucket='c218151a5506212l17131779t1w715054373660-labbucket-ntokzfktykpe'
prefix='flight-xgb'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

`sagemaker.inputs.TrainingInput` 関数を使用して、トレーニング用および検証データセット用の `record_set` を作成します。

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
from sagemaker.image_uris import retrieve
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
sess = sagemaker.Session()
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

xgb = sagemaker.estimator.Estimator(container,
                                    role = sagemaker.get_execution_role(), 
                                    instance_count=1, 
                                    instance_type=instance_type,
                                    output_path=s3_output_location,
                                    sagemaker_session=sess)
xgb.set_hyperparameters(max_depth=5,
                        eta=0.2,
                        gamma=4,
                        min_child_weight=6,
                        subsample=0.8,
                        silent=0,
                        objective='binary:logistic',
                        eval_metric = "auc", 
                        num_round=100)

xgb.fit(inputs=data_channels)

新しいモデルのバッチトランスフォーマーを使用し、テストデータセットでモデルを評価します。

In [ ]:
batch_X = test.iloc[:,1:];
batch_X_file='batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = xgb.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

予測ターゲットとテストラベルを取得します。

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

定義されたしきい値に基づいて予測値を計算します。

**注:** 予測ターゲットはスコアになります。スコアはバイナリークラスに変換する必要があります。

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

`target_predicted` と `test_labels` の混同行列をプロットします。

In [ ]:
# Enter your code here

ROC 管理図をプロットします。

In [ ]:
# Enter your code here

### さまざまなしきい値を試す

**問題**: どれほどうまくモデルがテストセットを処理したかに基づき、どう結論づけできますか?

In [ ]:
#Enter your answer here

### ハイパーパラメータの最適化 (HPO)

In [ ]:
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

### You can spin up multiple instances to do hyperparameter optimization in parallel

xgb = sagemaker.estimator.Estimator(container,
                                    role=sagemaker.get_execution_role(), 
                                    instance_count= 1, # make sure you have a limit set for these instances
                                    instance_type=instance_type, 
                                    output_path='s3://{}/{}/output'.format(bucket, prefix),
                                    sagemaker_session=sess)

xgb.set_hyperparameters(eval_metric='auc',
                        objective='binary:logistic',
                        num_round=100,
                        rate_drop=0.3,
                        tweedie_variance_power=1.4)

hyperparameter_ranges = {'alpha': ContinuousParameter(0, 1000, scaling_type='Linear'),
                         'eta': ContinuousParameter(0.1, 0.5, scaling_type='Linear'),
                         'min_child_weight': ContinuousParameter(3, 10, scaling_type='Linear'),
                         'subsample': ContinuousParameter(0.5, 1),
                         'num_round': IntegerParameter(10,150)}

objective_metric_name = 'validation:auc'

tuner = HyperparameterTuner(xgb,
                            objective_metric_name,
                            hyperparameter_ranges,
                            max_jobs=10, # Set this to 10 or above depending upon budget and available time.
                            max_parallel_jobs=1)

In [ ]:
tuner.fit(inputs=data_channels)
tuner.wait()

<i class="fas fa-exclamation-triangle" style="color:red"></i> トレーニングジョブが終了するまで待ちます。25～30 分かかる場合があります。

**ハイパーパラメータの最適化ジョブを監視するには:**  

1. AWS マネジメントコンソールの **サービス** メニューで **Amazon SageMaker** をクリックします。 
2. [**トレーニング**] > [**ハイパーパラメータの調整ジョブ**] の順にクリックします。
3. ハイパーパラメータの調整ジョブそれぞれのステータス、そのジョブの目標メトリクス値、ログを確認できます。 

ジョブが正常に完了したことを確認します。

In [ ]:
boto3.client('sagemaker').describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name)['HyperParameterTuningJobStatus']

ハイパーパラメータの調整ジョブには、最もよく機能したモデルがあります。このモデルに関する情報は、調整ジョブから取得できます。

In [ ]:
sage_client = boto3.Session().client('sagemaker')
tuning_job_name = tuner.latest_tuning_job.job_name
print(f'tuning job name:{tuning_job_name}')
tuning_job_result = sage_client.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=tuning_job_name)
best_training_job = tuning_job_result['BestTrainingJob']
best_training_job_name = best_training_job['TrainingJobName']
print(f"best training job: {best_training_job_name}")

best_estimator = tuner.best_estimator()

tuner_df = sagemaker.HyperparameterTuningJobAnalytics(tuning_job_name).dataframe()
tuner_df.head()

推定器 `best_estimator` を使用し、データを使ってトレーニングします。

**ヒント:** 前回の XGBoost estimator.fit 関数を参照してください。

In [ ]:
# Enter your code here'

新しいモデルのバッチトランスフォーマーを使用し、テストデータセットでモデルを評価します。

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = best_estimator.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

予測ターゲットとテストラベルを取得します。

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

`target_predicted` と `test_labels` の混同行列をプロットします。

In [ ]:
# Enter your code here

ROC 管理図をプロットします。

In [ ]:
# Enter your code here

**問題**: さまざまなハイパーパラメータやハイパーパラメータ範囲を試してください。それらの変更によって、モデルは改善されますか?

## まとめ

これで、トレーニングとモデルの評価を少なくとも数回繰り返しました。それでは、このプロジェクトをまとめて以下のことを考えてみましょう。

- 学んだこと 
- 今後取るであろうステップの種類 (もう少し時間があると想定して)

以下のセルを使用して、これらの質問や他の関連する質問に回答してください:

1. モデルのパフォーマンスは、ビジネス目標を満たしていますか? 満たしていない場合、チューニングにさらに多くの時間をかけられるとしたら、別にやってみたいことが何かありますか?
2. データセット、特徴量、ハイパーパラメータを変更したのにしたがい、モデルはどの程度改善しましたか? このプロジェクト全体を通してどのようなタイプの手法を採用しましたか、また、モデルで最大の改善を生み出したのはどの手法でしたか?
3. このプロジェクト全体を通して、直面した最大の課題は何でしたか?
4. パイプラインの側面に関する質問のうち、わからずに無回答のものがありますか?
5. このプロジェクトを進める中で機械学習について学んだ最も重要なことを 3 つ挙げるなら、何でしょうか?

#### <span style="color: blue;">プロジェクトプレゼンテーション: この質問に対する自分の回答の要約もプロジェクトプレゼンテーションに含めます。プロジェクトプレゼンテーションのためのメモをすべてまとめ、発見事項をクラスに示す準備をします。</span>

In [ ]:
# Write your answers here